In [0]:
# Run this in a temporary Databricks notebook cell first

%pip install requests pandas

In [0]:
import requests
import pandas as pd
from datetime import datetime, timedelta

# ----------------------------
# CONFIG
# ----------------------------
FINNHUB_API_KEY = "d6tbsj9r01qhkb43ho60d6tbsj9r01qhkb43ho6g"
TICKERS = [
    "AAPL", "NVDA", "MSFT", "GOOGL", "AMZN",
    "AMD", "TSLA", "META", "INTC", "NFLX"
]

# Finnhub free docs mention 1 year of historical company news on free tier.
# Start with 1 year to keep it simple and aligned with free access.
DATE_TO = datetime.utcnow().date()
DATE_FROM = DATE_TO - timedelta(days=365)

RAW_NEWS_TABLE_NAME = "p2_raw_news"


def fetch_finnhub_company_news(ticker: str, date_from: str, date_to: str, api_key: str) -> pd.DataFrame:
    url = "https://finnhub.io/api/v1/company-news"
    params = {
        "symbol": ticker,
        "from": date_from,
        "to": date_to,
        "token": api_key
    }

    resp = requests.get(url, params=params, timeout=30)
    resp.raise_for_status()
    data = resp.json()

    if not isinstance(data, list) or len(data) == 0:
        return pd.DataFrame(columns=[
            "published_at", "Date", "Ticker", "headline", "summary", "source", "url"
        ])

    rows = []
    for item in data:
        # Finnhub returns Unix timestamp in "datetime"
        published_ts = item.get("datetime")
        published_at = pd.to_datetime(published_ts, unit="s", utc=True) if published_ts else pd.NaT

        rows.append({
            "published_at": published_at,
            "Date": published_at.normalize() if pd.notnull(published_at) else pd.NaT,
            "Ticker": ticker,
            "headline": item.get("headline"),
            "summary": item.get("summary"),
            "source": item.get("source"),
            "url": item.get("url"),
        })

    return pd.DataFrame(rows)


all_news = []

for ticker in TICKERS:
    print(f"Fetching news for {ticker} ...")
    df_ticker_news = fetch_finnhub_company_news(
        ticker=ticker,
        date_from=str(DATE_FROM),
        date_to=str(DATE_TO),
        api_key=FINNHUB_API_KEY
    )
    all_news.append(df_ticker_news)

news_df = pd.concat(all_news, axis=0, ignore_index=True)

# Clean up
news_df = news_df.drop_duplicates(subset=["Ticker", "published_at", "headline"]).reset_index(drop=True)

print("Rows fetched:", len(news_df))


In [0]:
print(news_df.head(10).to_string(index=False))

In [0]:
# Save to Databricks table
spark.createDataFrame(news_df) \
    .write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(RAW_NEWS_TABLE_NAME)

In [0]:
news_df.head(5)